스마트 컨트랙트 보안 시나리오를 “반증 가능한 테스트”로 자동 변환하는 파이프라인

검증(반증) 시나리오의  결정적 실행 보장
실제로 그 위험이 ‘불가능’함을 증명

연구 내용(TODO)
1. ThreatScenario Class 구체화 -> 반증 시나리오의 빌드 에러가 안나면서 검증 task에 부합한 Unit Test 생성
2. Feedback Prompt 구체화 -> 반증 시나리오의 검증 목적 도달 가능과 도달 불가능, 취약점 존재와 미존재에 대한 검증과 수정을 "잘" 하는 지 검사하는 것
1, 2를 잘 정의하여 생산성

and .. (심볼릭, 동적 분석 툴 등을 추가)

최종 산출물: 검증(반증) 시나리오의  결정적 실행 보장

# skeleton code

In [2]:
################################################################################
# 0. module imports and type hints
################################################################################
from __future__ import annotations

import copy, json, re, subprocess, textwrap
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple
from jinja2 import Template


In [4]:
################################################################################
# 1.  Enumerations
################################################################################

class Category(str, Enum):
    """High‑level threat categories. Extend freely as needed."""

    SPOOFING = "Spoofing"
    TAMPERING = "Tampering"
    DOS = "Denial of Service"
    REENTRANCY = "Reentrancy"
    ACCESS_CONTROL = "Access Control"
    ARITHMETIC = "Arithmetic"
    # Feel free to add more …


class Severity(str, Enum):
    """Rough CVSS‑style importance bucket (optional metadata)."""

    CRITICAL = "critical"
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"
    INFO = "info"


################################################################################
# 2.  Dataclass representations
##############################.##################################################

@dataclass
class HelperContract:
    """Schema for an auxiliary on‑chain contract used inside a test."""

    name: str
    instance_name: str
    constructor_args: List[str] = field(default_factory=list)
    definition_code: str = ""

    # --- util -----------------------------------------------------------------
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    @staticmethod
    def from_dict(d: Dict[str, Any]) -> "HelperContract":
        return HelperContract(
            name=d["name"],
            instance_name=d.get("instance_name", d["name"].lower()),
            constructor_args=list(d.get("constructor_args", [])),
            definition_code=d.get("definition_code", ""),
        )


@dataclass
class ThreatScenario:
    """Complete description of a single negative test‑case."""

    # --- core metadata --------------------------------------------------------
    id: str
    category: Category
    description: str
    precondition: str
    action: str
    expected: str

    # --- optional meta --------------------------------------------------------
    severity: Severity = Severity.MEDIUM
    tags: List[str] = field(default_factory=list)

    # --- forge‑template‑specific extras ---------------------------------------
    target_contract_name: Optional[str] = None
    target_contract_declaration: Optional[str] = None
    target_contract_instance_name: Optional[str] = None
    required_imports: List[str] = field(default_factory=list)
    setup_code: Optional[str] = None
    test_setup_code: Optional[str] = None
    action_function: Optional[str] = None
    action_code: Optional[str] = None
    expected_revert_selector: Optional[str] = None
    assertion_code: Optional[str] = None

    # --- helper contracts -----------------------------------------------------
    helper_contracts: List[HelperContract] = field(default_factory=list)

In [5]:
################################################################################
# 3. scenario generation
################################################################################
from scenarios import SCENARIOS

In [6]:
################################################################################
# 4.  FORGE_TEMPLATE
################################################################################
FORGE_TEMPLATE = textwrap.dedent("""\
// SPDX-License-Identifier: UNLICENSED
// Generated by Audit Agent for Scenario: {{ scenario.id }}
pragma solidity {{ compiler_version | default('^0.8.24') }};

import "forge-std/Test.sol";
{%- if scenario.required_imports %}
// Scenario-specific imports:
{% for imp in scenario.required_imports %}{{ imp }}
{% endfor %}
{%- endif %}

{% for helper in helper_contracts %}
{{ helper.definition_code }}
{% endfor %}

contract {{ test_contract_name }} is Test {
    // --- State Variables ---
    // Target contract instance (filled by generator)
    {{ target_contract_declaration }} // 예: TargetContract internal targetContract;

    // Mock contract instances (filled by generator)
    {% if scenario.mock_deployments_code -%}
    {{ scenario.mock_deployments_code | indent(4) }}
    {%- endif %}

    // --- Setup ---
    function setUp() public {
        // General setup logic (filled by generator)
        {% if scenario.setup_code -%}
        {{ scenario.setup_code | indent(8) }}
        {%- else -%}
        // Default setup: Deploy target contract?
        // targetContract = new TargetContract();
        {%- endif %}
        
        console.log("Executing setUp for {{ test_contract_name }}");
    }

    // --- Test Function ---
    // MCP Test for Scenario: {{ scenario.id }} - {{ scenario.category }}
    // Description: {{ scenario.description }}
    function test_{{ scenario_id_snake_case }}() public {
        console.log(unicode"Scenario: {{ scenario.id }} - Precondition: {{ scenario.precondition }}");

        // --- Test Specific Setup ---
        {% if scenario.test_setup_code -%}
        {{ scenario.test_setup_code | indent(8) }}
        {%- else -%}
        // No specific test setup provided for this scenario.
        {%- endif %}
        
        console.log("Applying test-specific setup for {{ scenario.id }}...");

        // --- Execution & Assertion ---
        console.log("Action: {{ scenario.action }}"); // 설명용
        console.log("Expected: {{ scenario.expected }}"); // 설명용

        {% if scenario.expected_revert_selector -%}
        // Expect Revert
        vm.expectRevert({{ scenario.expected_revert_selector }});
        {% endif -%}

        // Execute the action (filled by generator)
        {% if scenario.action_code -%}
        {{ scenario.action_code | indent(8) }}
        {%- else -%}
        // Placeholder: Execute the action described in scenario.action
        // Example: {{ target_contract_instance_name | default('targetContract') }}.{{ scenario.action_function | default('someAction') }}(...);
        {%- endif %}
        
        console.log("Executing action for {{ scenario.id }}...");

        {% if not scenario.expected_revert_selector and scenario.assertion_code -%}
        // Assert State Changes (filled by generator)
        {{ scenario.assertion_code | indent(8) }}
        {%- elif not scenario.expected_revert_selector -%}
        // Default assertion if no specific assertion code is provided and no revert is expected
        assertTrue(true, "Execution completed without revert (no specific assertions)");
        {%- endif %}
    }
}
                                 
""")

def render_test(scenario: Dict[str, Any]) -> Tuple[str, str]:
    """
    시나리오 딕셔너리를 입력 받아 Jinja2 템플릿을 통해 Solidity 테스트 코드 생성.
    Returns: (파일 이름, 생성된 Solidity 소스코드 문자열)
    """
    test_contract_name = f"MCPTest_{scenario['id'].replace('-', '_').replace('.', '_')}"
    scenario_id_snake_case = re.sub(r'[^a-zA-Z0-9_]', '_', scenario['id']).lower()

    context = {
        "scenario": scenario,
        "test_contract_name": test_contract_name,
        "scenario_id_snake_case": scenario_id_snake_case,
        "compiler_version": scenario.get("compiler_version", "^0.8.24"),
        "target_contract_name": scenario.get("target_contract_name", "TargetContract"),
        "target_contract_declaration": scenario.get("target_contract_declaration", "// Target contract declaration missing"),
        "target_contract_instance_name": scenario.get("target_contract_instance_name", "targetContract"),
        "required_imports": scenario.get("required_imports", []),
        "setup_code": scenario.get("setup_code", ""),
        "test_setup_code": scenario.get("test_setup_code", ""),
        "action_function": scenario.get("action_function"),
        "action_code": scenario.get("action_code", ""),
        "expected_revert_selector": scenario.get("expected_revert_selector"),
        "assertion_code": scenario.get("assertion_code", "assertTrue(true, 'Default assertion');"),
        "helper_contracts": scenario.get("helper_contracts", []),
    }

    template = Template(FORGE_TEMPLATE, trim_blocks=True, lstrip_blocks=True)
    rendered_code = template.render(context)
    filename = f"{test_contract_name}.t.sol"
    return filename, rendered_code

In [7]:
import os
import shutil
from git import Repo

# GitHub 리포지토리 클론
github_url = "https://github.com/Uniswap/v4-core.git"
target_dir = "./v4-core"

if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

try:
    Repo.clone_from(github_url, target_dir)
    print(f"리포지토리 클론 완료: {target_dir}")
except Exception as e:
    print(f"클론 실패: {e}")
    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)

output_dir = "./v4-core/test/skeleton"
os.makedirs(output_dir, exist_ok=True)

for scenario in SCENARIOS:
    filename, code = render_test(scenario)
    with open(os.path.join(output_dir, filename), "w", encoding="utf-8") as f:
        f.write(code)

리포지토리 클론 완료: ./v4-core


In [8]:
import subprocess
result = subprocess.run(
    ["forge", "test", "--match-contract", "MCPTest_S_1_1", "-vvvv", "--json"],
    cwd="./v4-core",
    capture_output=True,
    text=True
)

# 테스트 결과 출력
print("테스트 실행 결과:")
print(result.stdout)


테스트 실행 결과:
Missing dependencies found. Installing now...

Updating dependencies in /Users/ham-yunsig/Documents/github/audit_agent/research/v4-core/lib
{"test/skeleton/MCPTest_S_1_1.t.sol:MCPTest_S_1_1":{"duration":"7ms 562us 292ns","test_results":{"test_s_1_1()":{"status":"Failure","reason":"Error != expected error: InvalidHookResponse() != custom error 0xe65af6a0","counterexample":null,"logs":[{"address":"0xf62849f9a0b5bf2913b396098f7c7019b51a820a","topics":["0x8be0079c531659141344cd1fd0a4f28419497f9722a3daafe3b4186f6b6457e0","0x0000000000000000000000000000000000000000000000000000000000000000","0x0000000000000000000000007fa9385be102ac3eac297483dd6233d62b3e1496"],"data":"0x"},{"address":"0x000000000000000000636f6e736f6c652e6c6f67","topics":["0x41304facd9323d75b11bcdd609cb38effffdb05710f7caf0e9b16c6d9d709f50"],"data":"0x00000000000000000000000000000000000000000000000000000000000000200000000000000000000000000000000000000000000000000000000000000021457865637574696e6720736574557020666f72204

In [9]:
import dataclasses
import json
import subprocess
import re # humantime_serde 파싱용
from typing import List, Dict, Optional, Any, Union, Tuple, Type, TypeVar
from dacite import from_dict, Config, DaciteError, UnexpectedDataError

T = TypeVar('T')

@dataclasses.dataclass
class TestStatus:
    # Rust: enum TestStatus { Success, Failure, Skipped }
    # JSON: "Success", "Failure", "Skipped" (단순 문자열로 처리)
    pass # 실제로는 TestResult.status가 문자열로 처리됨

@dataclasses.dataclass
class LogData:
    # Rust: alloy_primitives::Log
    # JSON: {"address": "...", "topics": [...], "data": "...", ...}
    address: str
    topics: List[str]
    data: str
    block_hash: Optional[str] = None
    block_number: Optional[int] = None # JSON에서는 문자열일 수 있음
    transaction_hash: Optional[str] = None
    transaction_index: Optional[int] = None # JSON에서는 문자열일 수 있음
    log_index: Optional[int] = None # JSON에서는 문자열일 수 있음
    removed: Optional[bool] = None

@dataclasses.dataclass
class BaseCounterExampleData:
    # Rust: fuzz::BaseCounterExample
    # JSON: {"calldata": "...", "sender": "..."} 등
    calldata: str # Hex string
    sender: Optional[str] = None
    # Rust 구조체에 더 많은 필드가 있을 수 있으나, JSON 출력 확인 필요
    # addr: Optional[str] = None
    # signature: Optional[str] = None
    # contract_name: Optional[str] = None

@dataclasses.dataclass
class CounterExampleData:
    # Rust: fuzz::CounterExample enum { Single(BaseCounterExample), Sequence(usize, Vec<BaseCounterExample>) }
    # JSON: {"Single": {...}} or {"Sequence": [original_len, [...]]}
    single_case: Optional[BaseCounterExampleData] = None
    sequence_original_len: Optional[int] = None
    sequence_cases: Optional[List[BaseCounterExampleData]] = None

# Simplified representation for complex nested structures
# 상세화 필요 시 별도 클래스 정의
TracesData = List[Tuple[str, Any]] # Vec<(TraceKind, CallTraceArena)> -> JSON: [["KindStr", {...TraceArena...}], ...]
BreakpointsData = Dict[str, Dict[str, List[str]]] # BTreeMap<Address, BTreeMap<usize, BTreeSet<BreakpointKind>>> -> JSON: {"0xAddr": {"pc_str": ["KindStr", ...]}}

@dataclasses.dataclass
class DurationData:
    # Rust: std::time::Duration
    # JSON: {"secs": u64, "nanos": u32}
    secs: int # JSON에서 문자열일 수 있음
    nanos: int # JSON에서 문자열일 수 있음

@dataclasses.dataclass
class TestKindUnit:
    # Rust: TestKind::Unit { gas: u64 }
    # JSON: {"Unit": {"gas": u64_or_str}}
    gas: int # JSON에서 문자열일 수 있음

@dataclasses.dataclass
class FuzzCaseData:
    # Rust: fuzz::FuzzCase
    # JSON: {"calldata": "...", "sender": "..."} 등 first_case 필드 내용
    calldata: str
    # 실제 FuzzCase 구조에 맞게 필드 추가/수정 필요
    # sender: Optional[str] = None

@dataclasses.dataclass
class TestKindFuzz:
    # Rust: TestKind::Fuzz { ... }
    # JSON: {"Fuzz": {"first_case": {...}, "runs": ..., "mean_gas": ..., "median_gas": ...}}
    first_case: FuzzCaseData
    runs: int
    mean_gas: int # JSON에서 문자열일 수 있음
    median_gas: int # JSON에서 문자열일 수 있음

@dataclasses.dataclass
class InvariantMetricsData:
    # Rust: invariant::InvariantMetrics
    # JSON: {"key": {...metrics...}}
    # 필요한 Metric 필드 정의 (예시)
    average_gas_cost: Optional[int] = None
    # ... 실제 Metric 필드 추가

@dataclasses.dataclass
class TestKindInvariant:
    # Rust: TestKind::Invariant { ... }
    # JSON: {"Invariant": {"runs": ..., "calls": ..., "reverts": ..., "metrics": {...}}}
    runs: int
    calls: int
    reverts: int
    metrics: Dict[str, InvariantMetricsData] # Map<String, InvariantMetrics>

# --- Core Result Structures ---

@dataclasses.dataclass
class TestResult:
    # Rust: struct TestResult
    status: str # TestStatus enum -> JSON: "Success", "Failure", "Skipped"
    reason: Optional[str]
    counterexample: Optional[CounterExampleData] # Rust Enum -> JSON: {"Single": ...} or {"Sequence": ...}
    logs: List[LogData]
    decoded_logs: List[str]
    kind: Union[TestKindUnit, TestKindFuzz, TestKindInvariant] # Rust Enum -> JSON: {"Unit": ...} or {"Fuzz": ...} or {"Invariant": ...}
    traces: TracesData
    labeled_addresses: Dict[str, str] # AddressHashMap<String>
    duration: DurationData # std::time::Duration -> JSON: {"secs": ..., "nanos": ...}
    breakpoints: BreakpointsData # evm::Breakpoints -> JSON: {"0xAddr": {"pc_str": [...]}}
    gas_snapshots: Dict[str, Dict[str, str]] # BTreeMap<String, BTreeMap<String, String>>

@dataclasses.dataclass
class SuiteResult:
    # Rust: struct SuiteResult
    # duration 필드는 humantime_serde 사용 -> JSON: "1.23s" 같은 문자열
    duration: str
    test_results: Dict[str, TestResult] # BTreeMap<String, TestResult>
    warnings: List[str]

# Top-level JSON structure: Dict[str, SuiteResult]
TestOutput = Dict[str, SuiteResult]

In [10]:
# --- Helper Functions for Parsing ---

def parse_int_field(data: Any) -> int:
    """JSON에서 문자열 또는 숫자로 올 수 있는 정수 필드를 파싱합니다."""
    if isinstance(data, str):
        return int(data)
    elif isinstance(data, int):
        return data
    raise TypeError(f"Expected int or string representation of int, got {type(data)}")

def parse_duration_from_string(duration_str: str) -> DurationData:
    """humantime_serde 형식의 문자열 ("1.23ms", "5s")을 DurationData로 변환 (단순화 버전)"""
    # 매우 단순한 파서 - 필요시 정교화
    secs = 0    
    nanos = 0
    try:
        num_str = re.findall(r"[\d\.]+", duration_str)[0]
        num = float(num_str)
        if 'ms' in duration_str:
            secs = int(num // 1000)
            nanos = int((num % 1000) * 1_000_000)
        elif 'µs' in duration_str or 'us' in duration_str:
             secs = int(num // 1_000_000)
             nanos = int((num % 1_000_000) * 1_000)
        elif 'ns' in duration_str:
             secs = int(num // 1_000_000_000)
             nanos = int(num % 1_000_000_000)
        elif 's' in duration_str:
            secs = int(num)
            nanos = int((num - secs) * 1_000_000_000)
        else: # 단위 없으면 초로 가정
            secs = int(num)
            nanos = int((num - secs) * 1_000_000_000)

    except (IndexError, ValueError) as e:
        print(f"Warning: Could not parse duration string '{duration_str}': {e}")
        # 기본값 또는 오류 처리

    return DurationData(secs=secs, nanos=nanos)


def dacite_from_dict(data_class: Type[T], data: Any) -> T:
    """dacite 파싱 래퍼 함수. 타입 훅 및 오류 처리 포함."""

    # 타입 훅 정의
    type_hooks = {
        # CounterExampleData 처리: Rust enum -> Python dataclass
        CounterExampleData: lambda d: CounterExampleData(single_case=dacite_from_dict(BaseCounterExampleData, d['Single']))
                                   if isinstance(d, dict) and 'Single' in d else
                                   CounterExampleData(sequence_original_len=int(d['Sequence'][0]),
                                                      sequence_cases=[dacite_from_dict(BaseCounterExampleData, case) for case in d['Sequence'][1]])
                                   if isinstance(d, dict) and 'Sequence' in d and isinstance(d.get('Sequence'), list) and len(d['Sequence']) == 2 else
                                   None,

        # TestKind 처리: Rust enum -> Python dataclass
        Union[TestKindUnit, TestKindFuzz, TestKindInvariant]: lambda d: dacite_from_dict(TestKindUnit, d['Unit'])
                                                                    if isinstance(d, dict) and 'Unit' in d else
                                                                    dacite_from_dict(TestKindFuzz, d['Fuzz'])
                                                                    if isinstance(d, dict) and 'Fuzz' in d else
                                                                    dacite_from_dict(TestKindInvariant, d['Invariant'])
                                                                    if isinstance(d, dict) and 'Invariant' in d else
                                                                    TestKindUnit(gas=0), # 기본값 또는 오류 처리

        # 기본 타입 변환 (JSON 문자열 -> Python int 등)
        int: parse_int_field,
        # TestResult.duration 필드는 dict 형태 {"secs": ..., "nanos": ...}
        DurationData: lambda d: from_dict(DurationData, d) if isinstance(d, dict) else DurationData(0, 0), # Dict만 처리
        str: lambda x: str(x) if x is not None else "", # None 방지 및 빈 문자열 처리
        # Optional[str]: lambda x: str(x) if x is not None else None, # 필요시 Optional[str] 명시적 처리
        list: lambda l: list(l) if l is not None else [], # None 방지
        dict: lambda di: dict(di) if di is not None else {}, # None 방지

        # 필요에 따라 다른 타입 훅 추가 (e.g., 복잡한 TracesData, BreakpointsData 파싱)
        # BreakpointsData: lambda d: {k: {str(k2): v2 for k2, v2 in v.items()} for k, v in d.items()} if isinstance(d, dict) else {} # 예시
    }

    config = Config(
        type_hooks=type_hooks,
        check_types=False, # 유연한 파싱 허용
        strict=False # 예상 못한 필드 무시
    )

    try:
        # dacite는 데이터 클래스 자체를 대상으로 파싱합니다.
        # TestResult나 SuiteResult 같은 클래스에 대한 전처리는 여기보다는
        # 이 함수를 호출하는 쪽(parse_forge_test_json)에서 수행하는 것이 더 적절할 수 있습니다.
        # 또는, 특정 클래스에 대한 커스텀 파서를 만들 수 있습니다.
        # 여기서는 일반적인 타입 변환 훅만 사용합니다.
        return from_dict(data_class=data_class, data=data, config=config)

    except DaciteError as e:
        print(f"Dataclass conversion error for {data_class.__name__}: {e}")
        # print(f"Problematic data field: {e.field_path}") # 오류 필드 확인
        # print(f"Problematic data value: {e.value}")     # 오류 값 확인
        # print(f"Problematic data context: {data}")      # 전체 데이터 확인
        raise
    except Exception as e:
        print(f"Unexpected error during parsing for {data_class.__name__}: {e}")
        # print(f"Data context: {data}")
        raise


def parse_forge_test_json(json_output: str) -> TestOutput:
    """forge test --json 결과를 파싱하여 데이터 클래스 객체로 변환합니다."""
    try:
        # JSON Lines 형식이면 각 라인을 파싱, 아니면 전체를 파싱
        lines = json_output.strip().splitlines()
        parsed_data: Dict[str, Any] = {} # 타입 힌트 명시

        if len(lines) > 1 and all(line.startswith('{') and line.endswith('}') for line in lines):
             # JSON Lines 처리 (예: forge script --json)
             # 여기서는 여러 JSON 객체가 각기 다른 스위트 결과를 나타낼 수 있다고 가정
             # 하지만 일반적으로 forge test --json은 단일 JSON 객체를 출력
             print("Warning: Multiple JSON objects detected (JSON Lines?). Attempting to merge.")
             merged_data = {}
             for line in lines:
                 try:
                     line_data = json.loads(line)
                     merged_data.update(line_data) # 키가 겹치면 덮어씀
                 except json.JSONDecodeError as e:
                     print(f"Skipping invalid JSON line: {e}")
                     continue
             parsed_data = merged_data
        elif lines:
             # 단일 JSON 객체 처리
             parsed_data = json.loads(lines[0]) # 첫 번째 라인 (또는 전체) 파싱
        else:
            print("Warning: JSON output is empty.")
            return {}


        # dacite를 사용하여 최상위 구조 파싱 (Dict[str, SuiteResult])
        # 각 스위트 결과를 개별적으로 파싱
        final_results: TestOutput = {}
        for suite_name, suite_data in parsed_data.items():
            try:
                # SuiteResult 파싱 시도
                # SuiteResult.duration은 문자열이므로 DurationData 훅이 아닌 기본 str 처리됨
                parsed_suite = dacite_from_dict(
                    data_class=SuiteResult,
                    data=suite_data
                )

                # TestResult 내부의 복잡한 필드(kind, counterexample 등)는
                # dacite_from_dict 내부의 훅에 의해 처리됨
                # SuiteResult 파싱 후 내부 TestResult 검증 또는 추가 처리 가능
                if isinstance(parsed_suite, SuiteResult):
                     for test_name, test_result in parsed_suite.test_results.items():
                          # 여기서 TestResult 내부의 kind나 counterexample 등을
                          # 필요하다면 추가 검증하거나 후처리할 수 있습니다.
                          # 예: kind가 None이면 기본값 설정 등
                          if test_result.kind is None:
                               print(f"Warning: Test '{test_name}' in suite '{suite_name}' has null kind. Setting default.")
                               # 기본값 설정 또는 다른 로직 처리
                               # test_result.kind = TestKindUnit(gas=0) # 예시
                          pass # 필요한 후처리 추가

                     final_results[suite_name] = parsed_suite
                else:
                     print(f"Warning: Failed to parse suite '{suite_name}' into SuiteResult object.")


            except DaciteError as e:
                print(f"Error parsing suite '{suite_name}': {e}")
                # print(f"Suite data: {suite_data}") # 디버깅 시 데이터 확인
                continue # 다음 스위트로 진행
            except Exception as e:
                print(f"Unexpected error parsing suite '{suite_name}': {e}")
                # print(f"Suite data: {suite_data}") # 디버깅 시 데이터 확인
                continue # 다음 스위트로 진행

        return final_results

    except json.JSONDecodeError as e:
        print(f"JSON 파싱 오류: {e}")
        print("--- Raw Output Start ---")
        print(json_output[:1000] + "..." if len(json_output) > 1000 else json_output) # 일부 출력
        print("--- Raw Output End ---")
        raise
    except Exception as e:
        # 이 블록은 최상위 JSON 파싱 또는 final_results 구성 중 예외 처리
        print(f"최상위 데이터 구조 변환 중 오류 발생: {e}")
        raise


def parse_forge_test_json(json_output: str) -> TestOutput:
    """forge test --json 결과를 파싱하여 데이터 클래스 객체로 변환합니다."""
    try:
        # JSON Lines 형식이면 각 라인을 파싱, 아니면 전체를 파싱
        lines = json_output.strip().splitlines()
        parsed_data: Dict[str, Any] = {} # 타입 힌트 명시

        if not lines:
            print("Warning: JSON output is empty.")
            return {}

        # 일반적으로 forge test --json은 단일 JSON 객체를 출력합니다.
        # JSON Lines 처리는 예외적인 경우를 위한 것입니다.
        if len(lines) > 1 and all(line.startswith('{') and line.endswith('}') for line in lines):
             # JSON Lines 처리 시도 (각 라인이 독립적인 JSON 객체라고 가정)
             # 실제 forge test 출력 형식에 따라 이 부분은 조정될 수 있습니다.
             # 예를 들어, 각 라인이 {"suite_name": {...}} 형태일 수도 있고,
             # 아니면 여러 라인에 걸쳐 하나의 큰 JSON 객체가 나뉘어 있을 수도 있습니다.
             # 여기서는 각 라인이 {"contract.sol:ContractName": {...}} 형태의 키-값 쌍을
             # 포함하는 독립적인 JSON이라고 가정하고 병합합니다.
             print("Warning: Multiple JSON objects detected (JSON Lines?). Attempting to merge.")
             merged_data = {}
             for i, line in enumerate(lines):
                 try:
                     line_data = json.loads(line)
                     # 각 라인이 최상위 구조(Dict[str, SuiteResult])의 일부라고 가정하고 업데이트
                     if isinstance(line_data, dict):
                         merged_data.update(line_data)
                     else:
                         print(f"Warning: Skipping non-dict JSON line {i+1}.")
                 except json.JSONDecodeError as e:
                     print(f"Skipping invalid JSON line {i+1}: {e}")
                     continue
             parsed_data = merged_data
        elif lines:
             # 단일 JSON 객체 처리 (가장 일반적인 경우)
             try:
                 # 여러 줄에 걸쳐 하나의 JSON 객체가 있을 수 있으므로 join
                 full_json_string = "".join(lines)
                 parsed_data = json.loads(full_json_string)
             except json.JSONDecodeError as e:
                 print(f"Failed to parse single JSON object: {e}")
                 print("--- Raw Output Start ---")
                 print(json_output[:1000] + "..." if len(json_output) > 1000 else json_output)
                 print("--- Raw Output End ---")
                 raise
        else:
             # lines가 비어있는 경우 (위에서 처리했지만 명시적으로)
             print("Warning: JSON output is empty after stripping.")
             return {}

        # parsed_data가 dict 형태인지 확인
        if not isinstance(parsed_data, dict):
             print(f"Error: Parsed JSON is not a dictionary, but type {type(parsed_data)}. Cannot proceed.")
             # print(f"Parsed data: {parsed_data}") # 디버깅용
             return {}


        # dacite를 사용하여 최상위 구조 파싱 (Dict[str, SuiteResult])
        # 각 스위트 결과를 개별적으로 파싱
        final_results: TestOutput = {}
        for suite_name, suite_data in parsed_data.items():
            if not isinstance(suite_data, dict):
                print(f"Warning: Skipping non-dictionary data for suite '{suite_name}'.")
                continue # 다음 스위트로 진행

            try:
                # SuiteResult 파싱 시도
                # SuiteResult.duration은 문자열이므로 DurationData 훅이 아닌 기본 str 처리됨
                # 타입 훅은 dacite_from_dict 내부에서 적용됨
                parsed_suite = dacite_from_dict(
                    data_class=SuiteResult,
                    data=suite_data
                )

                # SuiteResult 파싱 후 내부 TestResult 검증 또는 추가 처리 가능
                if isinstance(parsed_suite, SuiteResult):
                     # 필요한 후처리 추가 (예: 누락된 필드 기본값 설정)
                     for test_name, test_result in parsed_suite.test_results.items():
                          if test_result.kind is None:
                               # kind 훅에서 기본값을 설정하므로 이 경우는 드물지만, 방어적으로 처리
                               print(f"Warning: Test '{test_name}' in suite '{suite_name}' has null kind after parsing. Consider setting a default in the dataclass or hook.")
                               # test_result.kind = TestKindUnit(gas=0) # 필요시 여기서 강제 설정

                     final_results[suite_name] = parsed_suite
                else:
                     # dacite_from_dict가 예외를 발생시키지 않고 None 등을 반환하는 경우 (훅 설정에 따라)
                     print(f"Warning: Failed to parse suite '{suite_name}' into SuiteResult object (returned type: {type(parsed_suite)}).")


            except DaciteError as e:
                print(f"Error parsing suite '{suite_name}' with dacite: {e}")
                # print(f"Suite data for '{suite_name}': {suite_data}") # 디버깅 시 데이터 확인
                continue # 다음 스위트로 진행
            except Exception as e:
                print(f"Unexpected error parsing suite '{suite_name}': {e}")
                # print(f"Suite data for '{suite_name}': {suite_data}") # 디버깅 시 데이터 확인
                continue # 다음 스위트로 진행

        return final_results

    except json.JSONDecodeError as e:
        # json.loads 자체에서 발생한 오류
        print(f"JSON decoding error: {e}")
        print("--- Raw Output Start ---")
        print(json_output[:1000] + "..." if len(json_output) > 1000 else json_output) # 일부 출력
        print("--- Raw Output End ---")
        raise
    except Exception as e:
        # 최상위 수준에서의 예외 처리 (예: final_results 구성 중 오류)
        print(f"Error during top-level processing of test results: {e}")
        raise

In [11]:

# --- 실행 코드 ---

# forge test 명령어 실행
try:
    print("Running forge test...")
    result = subprocess.run(
        ["forge", "test", "--match-contract", "MCPTest_S_1_1", "-vvvv", "--json"],
        cwd="./v4-core", # <---- 실제 프로젝트 경로로 수정하세요!
        capture_output=True,
        text=True,
        check=False # check=True로 하면 실패 시 바로 예외 발생
    )

    if result.returncode != 0:
        print(f"Forge test command failed with return code {result.returncode}")
        print("Stderr:")
        print(result.stderr)
        # 실패하더라도 JSON 출력이 있을 수 있으므로 파싱 시도
        if not result.stdout.strip():
             print("No JSON output to parse.")
             exit() # JSON 출력 없으면 종료


    # 테스트 결과 JSON 파싱
    print("\nForge test finished. Parsing JSON output...")
    if result.stdout:
        parsed_results = parse_forge_test_json(result.stdout)

        # 파싱된 결과 사용
        if parsed_results:
            print(f"\nSuccessfully parsed {len(parsed_results)} test suite(s).")
            first_suite_key = next(iter(parsed_results))
            first_suite = parsed_results[first_suite_key]
            print(f"\n--- Example: First Suite ({first_suite_key}) ---")
            print(f"Duration: {first_suite.duration}") # 문자열로 출력됨
            print(f"Number of tests: {len(first_suite.test_results)}")
            print(f"Warnings: {first_suite.warnings}")

            if first_suite.test_results:
                first_test_key = next(iter(first_suite.test_results))
                first_test = first_suite.test_results[first_test_key]
                print(f"\n--- Example: First Test in Suite ({first_test_key}) ---")
                print(f"Status: {first_test.status}")
                print(f"Kind: {type(first_test.kind).__name__} - {first_test.kind}")
                print(f"Duration (raw): {first_test.duration}")
                print(f"Counterexample: {first_test.counterexample}")
                print(f"Number of logs: {len(first_test.logs)}")
                # print(dataclasses.asdict(first_test)) # 전체 TestResult를 dict로 출력 (디버깅용)

        else:
            print("Parsing resulted in empty data.")
    else:
        print("No JSON output received from forge test.")


except FileNotFoundError:
     print("Error: 'forge' command not found. Make sure foundry is installed and in your PATH.")
except subprocess.CalledProcessError as e:
    print(f"Forge test command execution error: {e}")
    print("Stderr:")
    print(e.stderr)
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    # 상세 오류 로깅 또는 스택 트레이스 출력 추가 가능

Running forge test...
Forge test command failed with return code 1
Stderr:


Forge test finished. Parsing JSON output...

Successfully parsed 1 test suite(s).

--- Example: First Suite (test/skeleton/MCPTest_S_1_1.t.sol:MCPTest_S_1_1) ---
Duration: 628us 250ns
Number of tests: 1
Warnings: []

--- Example: First Test in Suite (test_s_1_1()) ---
Status: Failure
Kind: TestKindUnit - TestKindUnit(gas=893688)
Duration (raw): DurationData(secs=0, nanos=157875)
Counterexample: None
Number of logs: 6


In [12]:
def pretty_print_test_summary(suite_name: str, suite: SuiteResult, max_logs_preview: int = 2) -> str:
    """
    Pretty-prints detailed test result summary for human/LLM review.
    """
    lines = []
    lines.append(f"🧪 Test Suite: {suite_name}")
    lines.append(f"├─ Duration         : {suite.duration}")
    lines.append(f"├─ Number of Tests  : {len(suite.test_results)}")
    lines.append("├─ Warnings         :")
    if suite.warnings:
        for warn in suite.warnings:
            lines.append(f"│   - {warn}")
    else:
        lines.append("│   None")
    lines.append("└─ Test Results:")

    for test_name, result in suite.test_results.items():
        lines.append(f"\n  • Test: {test_name}")
        lines.append(f"    ├─ Status         : {result.status}")
        lines.append(f"    ├─ Kind           : {type(result.kind).__name__} - {result.kind}")
        gas_used = getattr(result.kind, 'gas', 'N/A') if hasattr(result.kind, 'gas') else 'N/A'
        lines.append(f"    ├─ Gas Used       : {gas_used}")
        lines.append(f"    ├─ Duration       : {result.duration.secs}s {result.duration.nanos}ns")
        if result.reason:
            lines.append(f"    ├─ ❗ Failure Reason: {result.reason}")
        else:
            lines.append(f"    ├─ Reason         : None")

        # Counterexample
        if result.counterexample:
            if result.counterexample.single_case:
                case = result.counterexample.single_case
                lines.append(f"    ├─ Counterexample : Single - calldata: {case.calldata[:20]}...")
            elif result.counterexample.sequence_cases:
                lines.append(f"    ├─ Counterexample : Sequence ({result.counterexample.sequence_original_len} total)")
                for i, case in enumerate(result.counterexample.sequence_cases[:3]):
                    lines.append(f"    │   [{i}] calldata: {case.calldata[:20]}...")
        else:
            lines.append("    ├─ Counterexample : None")

        # Logs
        lines.append(f"    ├─ Logs           : {len(result.logs)} raw logs")
        lines.append(f"    ├─ Decoded Logs   : {len(result.decoded_logs)} entries")
        if result.decoded_logs:
            for i, entry in enumerate(result.decoded_logs[:max_logs_preview]):
                lines.append(f"    │   [{i}] {entry[:80]}{'...' if len(entry) > 80 else ''}")

        # Traces
        lines.append(f"    ├─ Traces         : {'Yes' if result.traces else 'No'}")
        lines.append(f"    ├─ Breakpoints    : {len(result.breakpoints)} addresses")
        lines.append(f"    ├─ Gas Snapshots  : {len(result.gas_snapshots)} keys")
        lines.append(f"    └─ Labeled Addrs  : {len(result.labeled_addresses)}")

    return "\n".join(lines)


In [13]:
suite_name = "test/skeleton/MCPTest_S_1_1.t.sol:MCPTest_S_1_1"
print(pretty_print_test_summary(suite_name, parsed_results[suite_name]))

🧪 Test Suite: test/skeleton/MCPTest_S_1_1.t.sol:MCPTest_S_1_1
├─ Duration         : 628us 250ns
├─ Number of Tests  : 1
├─ Warnings         :
│   None
└─ Test Results:

  • Test: test_s_1_1()
    ├─ Status         : Failure
    ├─ Kind           : TestKindUnit - TestKindUnit(gas=893688)
    ├─ Gas Used       : 893688
    ├─ Duration       : 0s 157875ns
    ├─ ❗ Failure Reason: Error != expected error: InvalidHookResponse() != custom error 0xe65af6a0
    ├─ Counterexample : None
    ├─ Logs           : 6 raw logs
    ├─ Decoded Logs   : 5 entries
    │   [0] Executing setUp for MCPTest_S_1_1
    │   [1] Scenario: S-1.1 - Precondition: PoolKey의 hooks 주소가 유효하지 않은 권한 비트를 가짐 (예: 0x...FF...
    ├─ Traces         : Yes
    ├─ Breakpoints    : 0 addresses
    ├─ Gas Snapshots  : 0 keys
    └─ Labeled Addrs  : 0


# Feedback Prompt
1. 동적 실행 과정에서 제공되는 디버깅 정보에서 다양한 피처를 뽑기
2. 피처를 Tool로 제공하여 LLM이 Unit Test를 수정함에 있어서 활용할 수 있는 파이프라이닝

-> 유니테스트 수정

In [27]:
def pretty_print_parsed_results(parsed_results: TestOutput) -> None:
    """
    parsed_results는 parse_forge_test_json의 반환값(TestOutput)입니다.
    모든 SuiteResult와 그 내부의 TestResult를 계층적으로 출력합니다.
    """
    import json

    for suite_name, suite in parsed_results.items():
        print(f"{'='*80}")
        print(f"Suite: {suite_name}")
        print(f"{'-'*80}")
        print(f"  Duration: {suite.duration!r}")
        if suite.warnings:
            print(f"  Warnings ({len(suite.warnings)}):")
            for w in suite.warnings:
                print(f"    - {w}")
        else:
            print("  Warnings: None")
        print()

        for test_name, test in suite.test_results.items():
            print(f"  {'-'*76}")
            print(f"  Test: {test_name}")
            print(f"    Status   : {test.status}")
            if test.reason:
                print(f"    Reason   : {test.reason}")
            # DurationData -> secs, nanos
            dur = test.duration
            print(f"    Duration : {dur.secs}s {dur.nanos}ns")
            # Kind
            kind = test.kind
            kind_name = type(kind).__name__
            print(f"    Kind     : {kind_name}")
            # Unit / Fuzz / Invariant 세부 정보
            if kind_name == "TestKindUnit":
                print(f"      Gas: {kind.gas}")
            elif kind_name == "TestKindFuzz":
                print(f"      Runs      : {kind.runs}")
                print(f"      Mean gas  : {kind.mean_gas}")
                print(f"      Median gas: {kind.median_gas}")
                print(f"      First case calldata: {kind.first_case.calldata}")
            elif kind_name == "TestKindInvariant":
                print(f"      Runs   : {kind.runs}")
                print(f"      Calls  : {kind.calls}")
                print(f"      Reverts: {kind.reverts}")
                print(f"      Metrics:")
                for key, metrics in kind.metrics.items():
                    print(f"        - {key}: {metrics}")
            print()

            # Counterexample
            if test.counterexample:
                ce = test.counterexample
                print("    Counterexample:")
                if ce.single_case:
                    sc = ce.single_case
                    print(f"      Single -> calldata: {sc.calldata}, sender: {sc.sender}")
                else:
                    print(f"      Sequence length: {ce.sequence_original_len}")
                    for idx, sc in enumerate(ce.sequence_cases or []):
                        print(f"        [{idx}] calldata: {sc.calldata}, sender: {sc.sender}")
                print()

            # Logs
            print(f"    Logs ({len(test.logs)}):")
            for idx, log in enumerate(test.logs):
                print(f"      [{idx}] address={log.address}, topics={log.topics}, data={log.data}")
            if test.decoded_logs:
                print(f"    Decoded Logs ({len(test.decoded_logs)}):")
                for dl in test.decoded_logs:
                    print(f"      - {dl}")
            print()

            # Traces (detailed)
            if test.traces:
                print(f"    Traces ({len(test.traces)}):")
                for trace_kind, arena in test.traces:
                    print(f"      {trace_kind}:")
                    arena_json = json.dumps(
                        arena,
                        indent=8,
                        ensure_ascii=False,
                    )
                    for line in arena_json.splitlines():
                        print(f"        {line}")
                print()

            # Breakpoints
            if test.breakpoints:
                print("    Breakpoints:")
                for addr, pcs in test.breakpoints.items():
                    print(f"      {addr}:")
                    for pc_str, kinds in pcs.items():
                        print(f"        pc {pc_str}: {kinds}")
                print()

            # Gas snapshots
            if test.gas_snapshots:
                print("    Gas Snapshots:")
                for label, snap in test.gas_snapshots.items():
                    print(f"      {label}: {snap}")
                print()

        print("\n")  # 다음 suite 구분


In [28]:
parsed = parse_forge_test_json(result.stdout)
pretty_print_parsed_results(parsed)


Suite: test/skeleton/MCPTest_S_1_1.t.sol:MCPTest_S_1_1
--------------------------------------------------------------------------------
  Duration: '628us 250ns'
  Warnings: None

  ----------------------------------------------------------------------------
  Test: test_s_1_1()
    Status   : Failure
    Reason   : Error != expected error: InvalidHookResponse() != custom error 0xe65af6a0
    Duration : 0s 157875ns
    Kind     : TestKindUnit
      Gas: 893688

    Logs (6):
      [0] address=0xf62849f9a0b5bf2913b396098f7c7019b51a820a, topics=['0x8be0079c531659141344cd1fd0a4f28419497f9722a3daafe3b4186f6b6457e0', '0x0000000000000000000000000000000000000000000000000000000000000000', '0x0000000000000000000000007fa9385be102ac3eac297483dd6233d62b3e1496'], data=0x
      [1] address=0x000000000000000000636f6e736f6c652e6c6f67, topics=['0x41304facd9323d75b11bcdd609cb38effffdb05710f7caf0e9b16c6d9d709f50'], data=0x00000000000000000000000000000000000000000000000000000000000000200000000000000000000

In [ ]:
def pretty_print_errors(parsed_results: TestOutput) -> None:
    """
    parsed_results: parse_forge_test_json() 반환값(TestOutput)
    실패한 단위 테스트의 에러 위치(트레이스의 revert 지점)와 이유만 추려서 출력합니다.
    """
    for suite_name, suite in parsed_results.items():
        for test_name, test in suite.test_results.items():
            if test.status.lower() != "failure":
                continue

            # 1) 스위트·테스트 식별 & 실패 이유
            print(f"Suite: {suite_name}")
            print(f"Test : {test_name}")
            if test.reason:
                print(f"Reason: {test.reason}")

            # 2) console.log 중 'Expected:' 메시지 (기대한 에러)
            for msg in test.decoded_logs:
                if msg.startswith("Expected:"):
                    print(msg)
                    break

            # 3) revert 발생 지점 찾기
            for trace_kind, arena in test.traces:
                # arena는 노드 리스트
                for node in arena if isinstance(arena, list) else []:
                    tr = node.get("trace", {})
                    if tr.get("status") == "Revert":
                        depth   = tr.get("depth")
                        caller  = tr.get("caller")
                        addr    = tr.get("address")
                        data    = tr.get("data")
                        output  = tr.get("output")
                        gas_used= tr.get("gas_used")
                        print(
                            f"Revert @ depth {depth}: {trace_kind} → {addr}\n"
                            f"  caller : {caller}\n"
                            f"  data   : {data}\n"
                            f"  output : {output}\n"
                            f"  gas    : {gas_used}"
                        )
                        # 첫번째 revert 만 출력
                        break
                else:
                    continue
                break

            print()  # 테스트 구분


In [29]:
parsed_results = parse_forge_test_json(result.stdout)
pretty_print_errors(parsed_results)

Suite: test/skeleton/MCPTest_S_1_1.t.sol:MCPTest_S_1_1
Test : test_s_1_1()
Reason: Error != expected error: InvalidHookResponse() != custom error 0xe65af6a0
Expected: vm.expectRevert(Hooks.HookAddressNotValid.selector)

